In [1]:
import os
import cv2
import torch           #imports and character mapping
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

# Character set and mappings
chars = "ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789"
char2idx = {c: i for i, c in enumerate(chars)}
idx2char = {i: c for c, i in char2idx.items()}

In [2]:
import numpy as np

def preprocess_image(path, img_size=(128, 64)):
    gray = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if gray is None:
        raise ValueError(f"Failed to load image at {path}")

    # 1. Otsu's thresholding for binarization
    _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

    # 2. Remove horizontal lines (adjust kernel size as needed)
    horizontal_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (25, 1))
    detected_lines = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, horizontal_kernel, iterations=1)
    no_lines = cv2.subtract(thresh, detected_lines)

    # 3. Remove small noise with morphological opening
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (2, 2))
    opened = cv2.morphologyEx(no_lines, cv2.MORPH_OPEN, kernel, iterations=1)

    # 4. Median blur for further denoising
    denoised = cv2.medianBlur(opened, 3)

    # 5. Optional: Sharpen the image
    sharpen_kernel = np.array([[0, -1, 0], [-1, 5,-1], [0, -1, 0]])
    sharpened = cv2.filter2D(denoised, -1, sharpen_kernel)

    # 6. Resize to target size
    final = cv2.resize(sharpened, img_size)
    return final

In [3]:
# ...existing code...
from torch.utils.data import random_split

# Build labels_dict from filenames                  #dataset preparation
img_dir = "A:/Datasets/samples/"
labels_dict = {}
for fname in os.listdir(img_dir):
    if fname.endswith(".png"):
        label_str = ''.join([c for c in os.path.splitext(fname)[0] if c.isalnum()]).upper()
        label_idx = [char2idx[c] for c in label_str]
        labels_dict[fname] = label_idx

from torch.utils.data import Dataset

class OCRDataset(Dataset):
    def __init__(self, img_dir, labels_dict, img_size=(128, 64)):
        self.img_dir = img_dir
        self.labels_dict = labels_dict
        self.img_size = img_size
        self.filenames = list(labels_dict.keys())

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        img_name = self.filenames[idx]
        img_path = os.path.join(self.img_dir, img_name)
        # Preprocess image
        img = preprocess_image(img_path, self.img_size)
        img = torch.tensor(img, dtype=torch.float32).unsqueeze(0) / 255.0  # (1, H, W), normalized
        label = torch.tensor(self.labels_dict[img_name], dtype=torch.long)
        return img, label
    
# Create the full dataset
full_dataset = OCRDataset(img_dir, labels_dict)

# Split into train and val
total_size = len(full_dataset)
val_size = int(0.1 * total_size)  # 10% for validation
train_size = total_size - val_size

train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])


# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=True, num_workers=0)

In [4]:

    class CRNN(nn.Module):
        def __init__(self, num_classes, hidden_size=256):
            super(CRNN, self).__init__()
    
            # CNN feature extractor
            self.cnn = nn.Sequential(
                nn.Conv2d(1, 64, kernel_size=3, stride=1, padding=1),
                nn.ReLU(),
                nn.MaxPool2d(2, 2),  # H/2, W/2
    
                nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
                nn.ReLU(),
                nn.MaxPool2d(2, 2),  # H/4, W/4
            )
    
            # Dummy forward pass to infer feature size
            with torch.no_grad():
                dummy = torch.zeros(1, 1, 64, 128)   # (B,C,H,W)
                feat = self.cnn(dummy)              # (1,C,H',W')
                _, C, H, W = feat.size()
                self.rnn_input_size = C * H         # flatten across H
    
            # BiLSTM
            self.rnn = nn.LSTM(
                input_size=self.rnn_input_size,
                hidden_size=hidden_size,
                num_layers=2,
                bidirectional=True,
                batch_first=True
            )
    
            # Fully connected
            self.fc = nn.Linear(hidden_size * 2, num_classes)  # 2 for bidirectional
    
        def forward(self, x):
            B, C, H, W = x.size()  # (B,1,64,128)
    
            # CNN
            x = self.cnn(x)        # (B,C,H',W')
    
            # Reshape for RNN
            B, C, H, W = x.size()
            x = x.permute(0, 3, 1, 2)          # (B,W,C,H)
            x = x.contiguous().view(B, W, C*H) # (B,W,features)
    
            # RNN
            x, _ = self.rnn(x)    # (B,W,hidden*2)
    
            # FC
            x = self.fc(x)        # (B,W,num_classes)
            return x


   

In [5]:
device = "cuda" if torch.cuda.is_available() else "cpu"                 
num_classes = len(idx2char) + 1  # +1 for CTC blank
model = CRNN(num_classes=num_classes).to(device)
criterion = nn.CTCLoss(blank=0, zero_infinity=True)                          #CTC Loss, Optimizer, and Decoder
optimizer = optim.Adam(model.parameters(), lr=1e-3)

def ctc_greedy_decoder(logits, idx2char):
    preds = logits.argmax(2).cpu().numpy()
    results = []
    for seq in preds:
        prev = -1
        s = ""
        for p in seq:
            if p != prev and p != 0:
                s += idx2char[p]
            prev = p
        results.append(s)
    return results

In [ ]:
def train_model(model, train_loader, val_loader, idx2char, epochs=3, device="cpu"):
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            log_probs = logits.log_softmax(2).permute(1,0,2)
            input_lengths = torch.full((images.size(0),), logits.size(1), dtype=torch.long)
           # ...inside your train_model loop...
            target_lengths = torch.full((labels.size(0),), labels.size(1), dtype=torch.long)
            loss = criterion(log_probs, labels, input_lengths, target_lengths)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item() 
        avg_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch+1}/{epochs} | Loss: {avg_loss:.4f}")

        print("Validation set size:", len(val_loader.dataset))

        # Validation sample
        model.eval()
        with torch.no_grad():
            for val_images, val_labels in val_loader:
                val_images = val_images.to(device)
                logits = model(val_images)
                decoded = ctc_greedy_decoder(logits, idx2char)
                print("\nSample predictions:")
                for i in range(min(5, len(decoded))):
                    gt = "".join([idx2char[c.item()] for c in val_labels[i] if c.item() != 0])
                    print(f"GT: {gt} | Pred: {decoded[i]}")
                break

train_model(model, train_loader, val_loader, idx2char, epochs=15, device=device)

In [ ]:
def show_example(original_path, processed_path):
    orig = cv2.imread(original_path, cv2.IMREAD_GRAYSCALE)
    proc = cv2.imread(processed_path, cv2.IMREAD_GRAYSCALE)

    plt.figure(figsize=(10,4)) 
    plt.subplot(1,2,1)
    plt.title("Original")
    plt.imshow(orig, cmap="gray")
    plt.axis("off")

    plt.subplot(1,2,2)
    plt.title("Processed")
    plt.imshow(proc, cmap="gray")
    plt.axis("off")
    plt.show()
    

In [ ]:
print("Train samples:", len(train_dataset))
print("Val samples:", len(val_dataset))

Train samples: 936
Val samples: 103


In [ ]:
def sequence_accuracy(preds, targets, idx2char):
    correct = 0
    total = len(preds)
    for i in range(total):
        gt = "".join([idx2char[c.item()] for c in targets[i] if c.item() != 0])
        if preds[i] == gt:
            correct += 1
    return correct / total
